# Step 8. 🎯 도전 과제

본인이 관심 있는 도구를 직접 만들어보세요. **Pydantic 정형 반환 패턴**을 적용하면서.

```python
class MyResult(BaseModel):
    field1: str
    field2: int

@tool
def my_tool(arg: type) -> MyResult:
    """..."""
    return MyResult(field1="...", field2=42)
```

## 과제 A: 단위 변환기 (실무 빈도 ★★★★★)

길이·무게·온도 변환 도구. 반환값을 `ConversionResult` Pydantic 객체로.

**힌트**
```python
class ConversionResult(BaseModel):
    original_value: float
    original_unit: str
    converted_value: float
    converted_unit: str

@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft')
    """
    # 변환 로직
    ...
```

테스트: "180cm가 몇 미터야?", "10km는 몇 마일이야?"

In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ 로컬 Jupyter
✅ API 키 OK
✅ import 완료


In [30]:
def run_with_tools(question: str, tools: list, llm_obj=llm, max_steps: int = 5):
    """도구 호출이 끝날 때까지 반복 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    tool_map = {t.name: t for t in tools}
    messages = [HumanMessage(content=question)]

    for _ in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:          
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc["name"]].invoke(tc["args"])

            # 현재 convert_length는 실패할 때 문자열을 반환함
            if isinstance(result, str):
                return result

            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )

    return "최대 단계 수를 초과했습니다."

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


In [31]:
class ConversionResult(BaseModel):
    original_value: float
    original_unit: str
    converted_value: Optional[float] = None
    converted_unit: str
    error: Optional[str] = None

@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft')
    """
    # 변환 로직

    conversion_factors = {
        'm': 1.0,
        'cm': 100.0,
        'km': 0.001,
        'inch': 39.3701,
        'ft': 3.28084
    }

    if from_unit not in conversion_factors or to_unit not in conversion_factors:
        return ConversionResult(
            original_value=value,
            original_unit=from_unit,
            converted_unit=to_unit,
            error=(
                f"'{from_unit}'에서 '{to_unit}'로 변환할 수 없습니다. "
                "지원하는 단위는 m, cm, km, inch, ft입니다."
            ),
        )

    # 미터 단위로 변환
    value_in_meters = value / conversion_factors[from_unit]
    # 목표 단위로 변환
    converted_value = value_in_meters * conversion_factors[to_unit]
    return ConversionResult(
        original_value=value,
        original_unit=from_unit,
        converted_value=converted_value,
        converted_unit=to_unit
    )

In [32]:
q = ["180cm가 몇 미터야?", "10km는 몇 마일이야?",]
for question in q:
    answer = run_with_tools(question, [convert_length])
    print(f"질문: {question}\n답변: {answer}\n")

질문: 180cm가 몇 미터야?
답변: 180cm는 1.8미터입니다.

질문: 10km는 몇 마일이야?
답변: 최대 단계 수를 초과했습니다.

